# Clustering Applications in Finance & Banking

---

This notebook shows **three applications** of unsupervised learning in finance and banking. We start with the mechanics of k-means on synthetic data and then work through two business cases.

## Contents and time budget

| Part | Topic | Cells | Time |
|---|---|---|---|
| 1 | Clustering fundamentals on synthetic data | 12 | about 30 min |
| 2 | Credit risk: does segmenting SME borrowers improve a default model? | 29 | about 75 min |
| 3 | Company valuation: clustering to find comparable firms | 21 | about 45 min |

Plan for two sessions (Part 1 and 2, then Part 3), or one long one with a break after Part 2. The timings assume you read every cell and try the exercises at the end of each part.

## Learning objectives

By the end of this notebook you will be able to:

1. Run k-means and DBSCAN with scikit-learn and read the silhouette score and WCSS of a partition.
2. Choose the number of clusters from a silhouette and elbow table, and say why.
3. Set up a fair comparison between one global credit model and several cluster-specific models: one split, one scaler, one test set.
4. Read a results table honestly, including the case where the simple model wins.
5. Turn a k-means peer group into a multiples-based valuation with a median multiple and a range, and say when the peer group is too thin to use.

## Data files

Part 1 generates its own data. Parts 2 and 3 load three CSV files from the course repository:

- `data/borrower_companies.csv` (Part 2)
- `data/financialdata_original.csv` and `data/financialdata_extra.csv` (Part 3)

If a load cell fails with a 404, the file has not been added to the repository yet. Ask the instructor; nothing else in the notebook needs changing.

## Why clustering in finance?

Clustering finds groups in data that carry no label. In banking and finance this is used for customer segmentation (behaviour, profitability, risk), for risk management (borrower groups, fraud as outliers, market regimes), for portfolio construction (assets with similar return patterns) and for valuation (comparable companies). Parts 2 and 3 take the last two seriously: we ask whether the groups actually help, rather than assume they do.


## Setup

We use pandas and NumPy for data handling, matplotlib and seaborn for plots, scikit-learn for clustering and logistic regression, and scipy for the z-score. Everything is imported once here. The helper `cluster_kmeans` is also defined once and reused in all three parts: it fits k-means with `n_init=10` (ten random starts, keep the best) and a fixed seed, so that every run of this notebook gives the same partition.


In [ ]:
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import metrics, preprocessing
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, roc_curve, precision_score, recall_score,
                             classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings('ignore')
%matplotlib inline

RAW = 'https://raw.githubusercontent.com/umatter/EDFB/main/'


def cluster_kmeans(df, nclust):
    '''Fit k-means with nclust clusters. Returns (silhouette, WCSS, labels, centroids).

    silhouette: 1 = well separated, 0 = overlapping, -1 = points in the wrong cluster.
    WCSS (within-cluster sum of squares, kmeans.inertia_): lower means tighter clusters.
    '''
    kmeans = KMeans(n_clusters=nclust, n_init=10, random_state=0).fit(df)
    label = kmeans.labels_
    centroids = kmeans.cluster_centers_
    sil = metrics.silhouette_score(df, label, metric='euclidean', random_state=0)
    wcss = kmeans.inertia_
    return sil, wcss, label, centroids


---

# Part 1: Clustering Fundamentals

We start with **synthetic data**, where we know the answer, to build intuition before touching financial data.

### What you will learn
1. How k-means works and what its two evaluation numbers mean
2. How to choose the number of clusters K
3. When k-means fails, and what DBSCAN does instead

### Key concepts

**K-means** is a partition method. It places K centroids, assigns every point to the nearest centroid, moves each centroid to the mean of its points, and repeats. It minimises the within-cluster sum of squares (WCSS) and needs K up front.

**Silhouette score** (from -1 to +1): for each point, how much closer it is to its own cluster than to the nearest other cluster. Near +1 means well separated clusters, near 0 means overlapping boundaries, negative means points that probably sit in the wrong cluster.

**WCSS** always falls as K grows (more centroids, shorter distances), so we look for the "elbow" where the drop flattens, not for the minimum.

**DBSCAN** is a density method. It grows clusters from points that have at least `min_samples` neighbours within radius `eps`, labels sparse points as noise (-1), and does not need K.


In [ ]:
# Create four Gaussian blobs in two dimensions
from sklearn.datasets import make_blobs

X, y = make_blobs(n_samples=200, n_features=2, centers=4, cluster_std=1.6, random_state=50)
print('X shape:', X.shape)
print('first five rows of X:\n', X[:5].round(2))
print('true group of the first five points:', y[:5])


In [ ]:
# Plot the raw data (no cluster information)
plt.figure(figsize=(10, 5))
plt.scatter(X[:, 0], X[:, 1], s=40)
plt.title('Synthetic data: four blobs')
plt.show()


In [ ]:
# Run k-means with K=4 and plot the partition
sil, wcss, label, centroids = cluster_kmeans(X, 4)
print('Silhouette:', round(sil, 3))
print('WCSS:', round(wcss, 1))

plt.figure(figsize=(10, 5))
plt.scatter(X[:, 0], X[:, 1], c=label, cmap='Accent', s=40)
plt.scatter(centroids[:, 0], centroids[:, 1], c='black', s=200, marker='P', label='centroids')
plt.legend()
plt.show()


### Choosing K

We do not know K in a real application. The usual approach is to fit k-means for a range of K and look at the silhouette (we want a high value) and the WCSS (we want the elbow). The cell below does both and shows the partitions side by side.


In [ ]:
max_n_clusters = 7
rows = []

fig, ax = plt.subplots(math.ceil((max_n_clusters - 1) / 2), 2, figsize=(14, 14), constrained_layout=True)
ax = ax.flatten()
for i in range(max_n_clusters - 1):
    nclust = i + 2
    sil, wcss, label, centroids = cluster_kmeans(X, nclust)
    rows.append({'Clusters': nclust, 'Silhouette': sil, 'WCSS': wcss})

    ax[i].scatter(X[:, 0], X[:, 1], c=label, cmap='Accent', s=30)
    ax[i].scatter(centroids[:, 0], centroids[:, 1], c='black', s=200, marker='P')
    ax[i].set_title(f'K = {nclust}   silhouette = {sil:.3f}   WCSS = {wcss:.0f}', fontsize=14)
plt.show()

tab = pd.DataFrame(rows)
display(tab.round(3))


In [ ]:
# Silhouette and WCSS against K on one figure (the "elbow plot")
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(tab.Clusters, tab.Silhouette, 'x-', color='blue')
ax1.set_xlabel('Number of clusters', fontsize=14)
ax1.set_ylabel('Silhouette', fontsize=14, color='blue')

ax2 = ax1.twinx()
ax2.plot(tab.Clusters, tab.WCSS, 'x-', color='red')
ax2.set_ylabel('WCSS', fontsize=14, color='red')
plt.show()


**Reading the table.** The silhouette peaks at the K that separates the blobs best, and the WCSS curve flattens after the same K. When the two agree, the choice is easy. In Part 2 they will not agree so neatly.

### When k-means fails

**Before running the next two cells:** the data below are two interleaved half-moons. How do you think k-means with K=2 will split them? Sketch your guess.


In [ ]:
from sklearn.datasets import make_moons

X2, y2 = make_moons(200, noise=0.05, random_state=0)

plt.figure(figsize=(10, 5))
plt.scatter(X2[:, 0], X2[:, 1], s=40)
plt.title('Two moons')
plt.show()


In [ ]:
# k-means with K=2 on the moons
sil, wcss, label, centroids = cluster_kmeans(X2, 2)
plt.figure(figsize=(10, 5))
plt.scatter(X2[:, 0], X2[:, 1], c=label, cmap='Accent', s=40)
plt.title('k-means, K=2')
plt.show()
print('Silhouette:', round(sil, 3))
print('WCSS:', round(wcss, 1))


In [ ]:
# DBSCAN: points with at least min_samples neighbours within radius eps start a cluster;
# the cluster grows through neighbours of neighbours; points in sparse regions become noise (-1)
db = DBSCAN(eps=0.3, min_samples=5).fit(X2)
label = db.labels_

plt.figure(figsize=(10, 5))
plt.scatter(X2[:, 0], X2[:, 1], c=label, cmap='Accent', s=40)
plt.title('DBSCAN, eps=0.3')
plt.show()
print('clusters found:', len(set(label)) - (1 if -1 in label else 0), '  noise points:', (label == -1).sum())


**What happened.** K-means cut the moons with a straight line, because each point goes to the nearest centroid and that boundary is always a straight line. Its silhouette is still positive: the metric measures compactness and separation of the partition it was given, not whether the partition matches the true shapes. DBSCAN followed the density along each moon and recovered both, without being told K. The price is two parameters (`eps`, `min_samples`) that have to fit the density of the data; try `eps=0.1` and `eps=0.6` to see the two failure modes.

### Exercises, Part 1

1. **Re-read the elbow table.** Change `cluster_std` in `make_blobs` from 1.6 to 3.0 and rerun the K sweep. Deliverable: the K with the highest silhouette before and after, and one sentence on why more overlap makes the choice of K less clear.
2. **DBSCAN parameters.** On the moons, run DBSCAN with `eps` in {0.1, 0.2, 0.3, 0.5} and `min_samples=5`. Deliverable: a four-row table with the number of clusters and the number of noise points for each `eps`, and one sentence on which `eps` you would pick and why.
3. **Silhouette is not an oracle.** Compute the silhouette of the DBSCAN labels on the moons (exclude noise points) and compare it with the k-means silhouette. Deliverable: both numbers and one sentence on why the visually correct partition can have the lower score.


---

# Part 2: Credit Risk Assessment via Clustering

> **Data file for this part:** `data/borrower_companies.csv`, loaded from the course repository. This file is not yet in the repository; the instructor must add it under `data/` before this part can run. The code assumes one row per loan applicant, a binary column `status` (0 = paid back, 1 = default), and every other column a numeric financial ratio.

## Business context

**Problem.** A P2P lending platform needs to assess the default risk of SME (small and medium enterprise) loan applicants.

**Challenge.** SMEs are heterogeneous. A trading company, a capital-intensive manufacturer and a young growth firm have different balance sheets, and the ratio that signals distress in one may be normal in another. One model for all of them may perform poorly.

**Idea.** Use clustering to find groups of similar borrowers, then fit one credit model per group.

## Dataset

`data/borrower_companies.csv`: SME loan applicants from a P2P lending platform, with financial ratios extracted from their annual statements (liquidity, profitability, leverage, efficiency) and a target column `status` (0 = paid back, 1 = default). The code below treats **every column except `status`** as a feature, so it does not depend on the exact ratio names.

**Decision-time rule.** At the moment the platform decides on a loan, which columns are already known? The ratios come from the financial statements the applicant submits with the application, so they are known. `status` is the outcome of the loan, known only years later, so it is the target and must never enter the clustering or the features.

## Methodology

**Phase 1, unsupervised.** Clean the data, split it once into a training and a test set, fit k-means on the training rows, and describe the segments.

**Phase 2, supervised.** Fit one logistic regression on all training rows (the global model) and one per cluster (the cluster models), and evaluate every model on the **same test rows**.

## The question, and how we will decide it

Does segmenting borrowers before modelling improve default prediction?

The argument for: different SME types have different default drivers, and a per-segment model can use segment-specific coefficients. The argument against: each cluster model sees only a fraction of the data, and logistic regression with fewer rows has noisier coefficients. Segmentation helps only if the coefficients really differ across segments by more than that noise costs.

**Success criterion, fixed before we look at any result:** the cluster models beat the global model if their pooled AUC on the common test set is higher by **more than 0.02**. A smaller gain is within the noise of a test set of this size and counts as "no improvement".


### Step 2.1: Load and explore the data


In [ ]:
dataset = pd.read_csv(RAW + 'data/borrower_companies.csv')
print(dataset.shape)
dataset.head()


In [ ]:
# Column types, missing values and the target
print(dataset.dtypes)
print('\nany missing values:', dataset.isna().any().any())
print('\ndefault rate:', round(dataset['status'].mean(), 4),
      '  (', int(dataset['status'].sum()), 'defaults of', len(dataset), ')')


In [ ]:
dataset.describe().T


### Step 2.2: Outliers

Financial ratios have heavy tails: a firm with almost no equity has a huge debt-to-equity ratio, and one such value can dominate the mean and the standard deviation of a column. K-means works with squared distances, so a few extreme rows can pull a centroid towards themselves and form a "cluster" of three companies.

The box plot below shows every standardised feature. We then flag rows whose z-score (distance from the mean in standard deviations) exceeds 4 on any feature.

Two points to keep in mind:

- `status` is excluded from the z-score. For a binary column with default rate *p*, a 1 has z-score sqrt((1-p)/p), which exceeds 4 whenever *p* is below 5.9 %; including it would delete the defaulters.
- Dropping rows is the crude option. It removes exactly the firms with extreme leverage or liquidity, which is the tail a default model exists to capture. We print the default rate before and after so you can see what the filter cost. Winsorising (capping each ratio at its 1st and 99th percentile) keeps the rows; Exercise 2.1 asks you to try it.


In [ ]:
def box_plot(df, standardize=True, title=''):
    '''Horizontal box plot of every column of df (standardised by default so scales are comparable).'''
    plt.figure(figsize=(12, 7))
    if standardize:
        df = pd.DataFrame(preprocessing.StandardScaler().fit_transform(df.values), columns=df.columns)
    long = pd.melt(df.reset_index(), id_vars='index', value_vars=list(df.columns))
    ax = sns.boxplot(x='value', y='variable', data=long, orient='h')
    ax.tick_params(labelsize=10)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_title(title + '\n(variables standardised for comparability)', fontsize=14)
    plt.show()


box_plot(dataset.drop(columns='status'), title='Before outlier filter')


In [ ]:
from scipy import stats

features = dataset.drop(columns='status')
z = np.abs(stats.zscore(features))            # status is NOT included
keep = (z < 4).all(axis=1)
dataset_o = dataset[keep].copy()

print('rows before:', len(dataset), '  rows after:', len(dataset_o), '  dropped:', (~keep).sum())
print('default rate before:', round(dataset['status'].mean(), 4),
      '  after:', round(dataset_o['status'].mean(), 4))
print('defaulters dropped:', int(dataset.loc[~keep, 'status'].sum()), 'of', int(dataset['status'].sum()))


In [ ]:
box_plot(dataset_o.drop(columns='status'), title='After outlier filter')


### Step 2.3: One split, then clustering on the training rows

We now split `dataset_o` **once** into a training set (80 %) and a test set (20 %), stratified on `status` so both halves have the same default rate. Everything that is fitted from here on, the scaler, k-means and every logistic regression, is fitted on the training rows only. Test rows are transformed with the training scaler and assigned to a cluster with `kmeans.predict`. That is exactly what would happen to a new applicant in production, and it keeps the test set a clean measure of out-of-sample performance.

The index of `dataset_o` is kept on the scaled frames so features and `status` can never be mismatched.


In [ ]:
train_df, test_df = train_test_split(dataset_o, test_size=0.2, random_state=0, stratify=dataset_o['status'])

feature_cols = [c for c in dataset_o.columns if c != 'status']
scaler = preprocessing.StandardScaler().fit(train_df[feature_cols])

X_train = pd.DataFrame(scaler.transform(train_df[feature_cols]), columns=feature_cols, index=train_df.index)
X_test = pd.DataFrame(scaler.transform(test_df[feature_cols]), columns=feature_cols, index=test_df.index)
y_train = train_df['status']
y_test = test_df['status']

assert (X_train.index == y_train.index).all() and (X_test.index == y_test.index).all()
assert y_train.notna().all() and y_test.notna().all()
print('train:', X_train.shape, ' default rate', round(y_train.mean(), 4))
print('test: ', X_test.shape, ' default rate', round(y_test.mean(), 4))


**Visualising many dimensions.** We cannot plot the borrowers directly, so we use principal component analysis (PCA) to project the standardised features onto their two directions of largest variance. The clusters are still fitted in the full feature space; PCA is only used for the pictures.


In [ ]:
pca = PCA(random_state=0).fit(X_train)
scores_train = pca.transform(X_train)

exp_var = pca.explained_variance_ratio_
print('variance explained by the first two components:', round(exp_var[:2].sum(), 3))

plt.figure(figsize=(10, 5))
plt.bar(range(1, len(exp_var) + 1), exp_var, alpha=0.5, label='individual')
plt.step(range(1, len(exp_var) + 1), np.cumsum(exp_var), where='mid', label='cumulative')
plt.xlabel('Principal component', fontsize=13)
plt.ylabel('Explained variance ratio', fontsize=13)
plt.legend()
plt.show()


In [ ]:
# K sweep on the training rows: partition (in PC space), silhouette and WCSS side by side
max_n_clusters = 7
rows = []
label_list = {}

fig, ax = plt.subplots(math.ceil((max_n_clusters - 1) / 2), 2, figsize=(14, 16), constrained_layout=True)
ax = ax.flatten()
for i in range(max_n_clusters - 1):
    nclust = i + 2
    sil, wcss, label, _ = cluster_kmeans(X_train, nclust)
    rows.append({'Clusters': nclust, 'Silhouette': sil, 'WCSS': wcss})
    label_list[nclust] = label
    cent = pd.DataFrame(scores_train).groupby(label).mean().values

    ax[i].scatter(scores_train[:, 0], scores_train[:, 1], c=label, cmap='Accent', s=15)
    ax[i].scatter(cent[:, 0], cent[:, 1], c='black', s=200, marker='P')
    ax[i].set_title(f'K = {nclust}   silhouette = {sil:.3f}   WCSS = {wcss:.0f}', fontsize=14)
    ax[i].set_xlabel('PC1'); ax[i].set_ylabel('PC2')
plt.show()

tab = pd.DataFrame(rows)
display(tab.round(3))


In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(tab.Clusters, tab.Silhouette, 'x-', color='blue')
ax1.set_xlabel('Number of clusters', fontsize=14)
ax1.set_ylabel('Silhouette', fontsize=14, color='blue')
ax2 = ax1.twinx()
ax2.plot(tab.Clusters, tab.WCSS, 'x-', color='red')
ax2.set_ylabel('WCSS', fontsize=14, color='red')
plt.show()


**Choosing K.** On real financial ratios the silhouette is low for every K (values around 0.1 to 0.3 are typical: firms form a continuum, not tight islands) and the WCSS curve has no sharp elbow. We use **K = 3** as a compromise: K = 2 is a single split of the sample and tells us little about heterogeneity, while K of 4 or more leaves clusters with too few defaults to fit a separate model (check `defaults_test` in the results table below), and three segments can still be described in business terms. Look at the table from your run and check whether K = 3 is at or near the silhouette peak; if K = 2 is clearly better on your data, the comparison below still works, only with two cluster models.


In [ ]:
number_of_clusters = 3

kmeans = KMeans(n_clusters=number_of_clusters, n_init=10, random_state=0).fit(X_train)
train_label = pd.Series(kmeans.labels_, index=X_train.index, name='cluster')
test_label = pd.Series(kmeans.predict(X_test), index=X_test.index, name='cluster')

summary = pd.DataFrame({
    'n_train': train_label.value_counts().sort_index(),
    'default_rate_train': y_train.groupby(train_label).mean().round(3),
    'n_test': test_label.value_counts().sort_index(),
    'default_rate_test': y_test.groupby(test_label).mean().round(3),
})
summary.index.name = 'cluster'
display(summary)


### Step 2.4: What are the segments?

The figure below shows, for every feature, the distribution of standardised values in each cluster (training rows). A feature whose three curves sit on top of each other does not distinguish the clusters; a feature whose curves are shifted does. Read the shifted ones and give each cluster a name.


In [ ]:
X_labelled = X_train.copy()
X_labelled['cluster'] = train_label

ncol = 3
fig, ax = plt.subplots(math.ceil(len(feature_cols) / ncol), ncol, figsize=(16, 2.6 * math.ceil(len(feature_cols) / ncol)),
                       constrained_layout=True)
ax = ax.flatten()
for i, var in enumerate(feature_cols):
    for clust in range(number_of_clusters):
        sns.kdeplot(X_labelled.loc[X_labelled['cluster'] == clust, var], ax=ax[i], fill=True, alpha=0.3,
                    label=f'cluster {clust}', warn_singular=False)
    ax[i].set_title(var, fontsize=12)
    ax[i].set_xlabel(''); ax[i].set_ylabel('')
    ax[i].set_xlim(-3, 3)
ax[0].legend(fontsize=9)
for j in range(len(feature_cols), len(ax)):
    ax[j].axis('off')
plt.show()

# The same information as numbers: mean standardised value of each feature per cluster
display(X_labelled.groupby('cluster').mean().T.round(2))


**Naming the clusters.** Use the density panels and the table of cluster means above. For each cluster, find the two or three features that are shifted furthest from zero (a mean of +0.8 means the cluster sits almost one standard deviation above the average borrower on that ratio) and translate them into business language, for example:

- cluster 0: (describe from the densities, e.g. "high leverage, thin liquidity, low margins": the stressed segment)
- cluster 1: (describe from the densities, e.g. "average on most ratios, strong turnover": the mature trading segment)
- cluster 2: (describe from the densities, e.g. "high liquidity, high growth, low profitability": the young growth segment)

The placeholders are deliberate: this notebook was written without the data file, so the segment names must come from your run. Compare your names with the default rates per cluster in the table above. A segment you called "stressed" should have the highest default rate; if it does not, either the name or the intuition needs revisiting.


### Step 2.5: Credit models, global versus cluster-specific

The comparison is set up so that every model is judged on the **same test rows**:

- the **global model** is a logistic regression fitted on all training rows;
- each **cluster model** is a logistic regression fitted on the training rows of one cluster, and evaluated on the test rows that `kmeans.predict` assigned to that cluster;
- the **pooled cluster model** collects the cluster models' predicted probabilities over the whole test set, so it can be compared with the global model on identical rows.

Metrics, each with the baseline it must beat:

- **AUC** from the predicted probabilities (threshold-free; 0.5 is a coin flip). This is the headline number.
- **Recall** (share of actual defaults we flag) and **precision** (share of flagged loans that default) at the 0.5 threshold. On an unbalanced sample the 0.5 threshold flags very few loans, so these two numbers are reported for context and are not the decision criterion.
- The **majority-class baseline** (predict "no default" for everyone) has recall 0 and an AUC of 0.5.

We do **not** undersample. AUC does not depend on the class balance, and undersampling before the split (as older versions of this notebook did) would have measured every model on an artificial 33 % default population. If you want to see the effect of undersampling on recall, Exercise 2.2 does it on the training set only.

The whole model block is one function, so that the only thing that differs between the global run and the cluster runs is the rows passed in.


In [ ]:
def evaluate_credit_model(Xtr, ytr, Xte, yte, name):
    '''Fit a logistic regression on (Xtr, ytr), evaluate on (Xte, yte). Returns a dict of results.'''
    model = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=0).fit(Xtr, ytr)
    prob = model.predict_proba(Xte)[:, 1]
    pred = (prob >= 0.5).astype(int)
    return {'model': name,
            'n_train': len(Xtr), 'n_test': len(Xte),
            'defaults_test': int(yte.sum()),
            'AUC': roc_auc_score(yte, prob),
            'recall': recall_score(yte, pred, zero_division=0),
            'precision': precision_score(yte, pred, zero_division=0),
            'prob': pd.Series(prob, index=Xte.index),
            'fitted': model}


global_res = evaluate_credit_model(X_train, y_train, X_test, y_test, 'global')
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in global_res.items() if k not in ('prob', 'fitted')})
print('majority-class baseline: AUC 0.5, recall 0, test default rate', round(y_test.mean(), 3))


In [ ]:
# Which ratios drive the global model? Standardised features, so coefficients are comparable.
coef = pd.Series(global_res['fitted'].coef_[0], index=feature_cols).sort_values()
plt.figure(figsize=(8, 6))
coef.plot.barh()
plt.title('Global model: logistic regression coefficients (standardised features)')
plt.xlabel('coefficient (log-odds per one standard deviation)')
plt.show()


In [ ]:
# Confusion matrix and classification report of the global model at the 0.5 threshold
pred_global = (global_res['prob'] >= 0.5).astype(int)
ConfusionMatrixDisplay.from_predictions(y_test, pred_global, display_labels=['paid (0)', 'default (1)'], cmap='Blues')
plt.show()
print(classification_report(y_test, pred_global, zero_division=0))


In [ ]:
# ROC curve of the global model. The legend AUC is computed from the same probabilities as the curve.
fpr, tpr, _ = roc_curve(y_test, global_res['prob'])
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"global model (AUC = {global_res['AUC']:.3f})")
plt.plot([0, 1], [0, 1], 'r--', label='coin flip (AUC = 0.5)')
plt.xlabel('False positive rate'); plt.ylabel('True positive rate')
plt.title('ROC, global model'); plt.legend(loc='lower right')
plt.show()


In [ ]:
# Cluster-specific models: same function, one cluster's training rows in, that cluster's test rows out
rows = [{k: v for k, v in global_res.items() if k not in ('prob', 'fitted')}]
rows[0]['global_AUC_same_rows'] = global_res['AUC']
pooled_prob = pd.Series(index=X_test.index, dtype=float)

for k in range(number_of_clusters):
    tr = train_label == k
    te = test_label == k
    if y_train[tr].nunique() < 2 or y_test[te].nunique() < 2:
        print(f'cluster {k}: only one class in train or test rows, no model fitted')
        continue
    res = evaluate_credit_model(X_train[tr], y_train[tr], X_test[te], y_test[te], f'cluster {k}')
    pooled_prob[te[te].index] = res['prob']
    # the global model scored on exactly the same test rows as this cluster model
    res['global_AUC_same_rows'] = roc_auc_score(y_test[te], global_res['prob'][te])
    rows.append({k2: v for k2, v in res.items() if k2 not in ('prob', 'fitted')})

# Pooled cluster models: every test row scored by the model of its own cluster
mask = pooled_prob.notna()
pooled_pred = (pooled_prob[mask] >= 0.5).astype(int)
rows.append({'model': 'cluster models pooled',
             'n_train': len(X_train), 'n_test': int(mask.sum()), 'defaults_test': int(y_test[mask].sum()),
             'AUC': roc_auc_score(y_test[mask], pooled_prob[mask]),
             'recall': recall_score(y_test[mask], pooled_pred, zero_division=0),
             'precision': precision_score(y_test[mask], pooled_pred, zero_division=0),
             'global_AUC_same_rows': roc_auc_score(y_test[mask], global_res['prob'][mask])})

results = pd.DataFrame(rows).set_index('model')
results['AUC_gain_vs_global'] = results['AUC'] - results['global_AUC_same_rows']
display(results.round(3))

gain = results.loc['cluster models pooled', 'AUC_gain_vs_global']
print(f'\nPooled cluster models minus global model, AUC on the common test rows: {gain:+.3f}')
print('Success criterion (gain > 0.02):', 'MET' if gain > 0.02 else 'NOT MET')


In [ ]:
# Both ROC curves on the same test rows
fig, ax = plt.subplots(figsize=(6, 5))
for name, prob in [('global', global_res['prob'][mask]), ('cluster models pooled', pooled_prob[mask])]:
    fpr, tpr, _ = roc_curve(y_test[mask], prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc_score(y_test[mask], prob):.3f})')
ax.plot([0, 1], [0, 1], 'r--', label='coin flip')
ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate')
ax.set_title('ROC on the common test set'); ax.legend(loc='lower right')
plt.show()


### Reading the table

Read the results table row by row, in this order:

1. **Global model.** Its AUC is the number every other row has to beat. If it is below about 0.65, the ratios carry little signal and no segmentation will rescue that.
2. **One cluster at a time.** Compare `AUC` with `global_AUC_same_rows`: that is the global model scored on exactly the rows this cluster model was scored on. A cluster model with a positive `AUC_gain_vs_global` learned something segment-specific; a negative gain means the segment-specific coefficients were not worth the loss of training rows. Small clusters with few test defaults (`defaults_test` under 15 or so) have very noisy AUCs, so treat their gains as suggestive, not as evidence.
3. **Pooled row.** This is the decision. The pre-registered criterion was a gain above 0.02.

What the outcome means either way:

- **Gain above 0.02:** the segments have different default drivers, and the platform should consider segment-specific scorecards, with the caveat that new applicants must be assigned to a cluster with the same scaler and the same `kmeans` object, and that clusters drift over time.
- **Gain between -0.02 and 0.02:** clustering neither helped nor hurt. The global model already captures what the ratios have to say, and the extra complexity of three models is not paid for. This is the most common outcome with logistic regression on a few thousand rows.
- **Gain below -0.02:** the cluster models are worse. Each one was fitted on a third of the data and its coefficients are noisier; the global model's larger sample beat segment-specific fit.

On the synthetic stand-in data used to write this notebook (the real file was not available), the pooled gain was about -0.03, the third case: the global model won. Check the number printed above your table and write down which of the three cases your run falls in.

An honest lesson from this part: segmentation is a hypothesis about heterogeneous coefficients, not a free improvement. The way to find out is the comparison you just ran, on common test rows, with a criterion fixed before looking.


### Part 2 summary

- Outlier filtering must exclude the target and must report what it removed.
- One split, one scaler, one `kmeans` fitted on training rows, and every model judged on the same test rows: that is the minimum for a fair comparison of a global and a segmented model.
- AUC comes from predicted probabilities, and the baseline it must beat is the global model, not a coin flip.
- Cluster models trade segment-specific coefficients against smaller samples. Whether the trade pays is an empirical question with a pre-registered answer, not a slogan.

### Exercises, Part 2

1. **Winsorise instead of dropping.** Replace the z-score filter with capping every ratio at its 1st and 99th percentile (`df.clip(lower=df.quantile(0.01), upper=df.quantile(0.99), axis=1)`), rerun the split, the clustering and the results table. Deliverable: the default rate kept (it should now be the full-sample rate), the new pooled gain, and one sentence on whether the filter changed the conclusion.
2. **Undersampling on the training set only.** Inside `evaluate_credit_model`, before fitting, keep every training default and a random sample of non-defaults twice that size. Deliverable: the recall and precision of the global model before and after, and one sentence on why the AUC barely moves while recall does.
3. **Two clusters or five.** Rerun Step 2.5 with `number_of_clusters` equal to 2 and to 5. Deliverable: the three pooled gains (K = 2, 3, 5) and one sentence on how the number of test defaults per cluster limits what K can be used.


---

# Part 3: Company Valuation Using Clustering

> **Data files for this part:** `data/financialdata_original.csv` (fundamentals, one row per company, a text column `shortName` that includes `Company_11`, a numeric column `ebitda`, other numeric columns) and `data/financialdata_extra.csv` (market data keyed by `shortName`, with at least `enterpriseValue` and `marketCap`). Neither file is in the repository yet; the instructor must add both under `data/` before this part can run.

## Business context

**Problem.** You need to value a **private company** (Company_11) that is not publicly traded and therefore has no market price.

**Solution.** The **multiples method** with comparable companies:
1. Find public companies similar to Company_11.
2. Compute their valuation multiples (here EV/EBITDA).
3. Apply a representative multiple to Company_11's EBITDA.

**Role of clustering.** Peer selection is normally done by an analyst from industry codes and judgement, and it is the step where most of the subjectivity in a multiples valuation lives. Clustering on financial characteristics is an objective and repeatable alternative, and it is also the step we will look at most critically.

## The multiples method

- **EV/EBITDA**: enterprise value (market value of equity plus net debt) over earnings before interest, tax, depreciation and amortisation. Because EV includes debt and EBITDA is measured before interest, the multiple is neutral to capital structure: a firm financed with more debt does not look cheaper or dearer than an identical firm financed with equity. That is why we match EV with EBITDA and not market capitalisation with EBITDA.
- **P/E**: simple, but affected by leverage and by one-off items below the operating line.
- **EV/Sales**: usable for loss-making firms; says nothing about profitability. We use it as a cross-check.

### Formula

```
Company_11 EV = median(peer EV/EBITDA) x Company_11 EBITDA
```

The **median** of the peers' multiples is used, not the mean: one peer with a multiple of 60 would move the mean of a ten-firm group by 5 turns of EBITDA. We also report the interquartile range of the multiples as a low/high range for the valuation.

## Six-step process

1. Load fundamentals for a set of companies.
2. Clean the data (missing values).
3. Model selection: choose the number of clusters.
4. Cluster the companies.
5. Identify Company_11's cluster; check the peer group is large enough.
6. Compute the peers' multiples and value Company_11.

## Data files

`data/financialdata_original.csv` holds fundamentals (income statement and balance sheet items, margins, growth) with one row per company and a `shortName` column. `data/financialdata_extra.csv` holds market data (`marketCap`, `enterpriseValue`) for the public companies, keyed by `shortName`.


### Step 3.1 and 3.2: Load and clean


In [ ]:
dataset = pd.read_csv(RAW + 'data/financialdata_original.csv')
print(dataset.shape)
dataset.head(12)


In [ ]:
print(dataset.dtypes)
print('\nmissing values per column (only columns with any):')
print(dataset.isna().sum()[dataset.isna().sum() > 0])
dataset.describe().T


**Missing values.** Rows with missing fundamentals cannot be clustered, so we drop them, with one exception: Company_11 is the company we want to value, so it must stay in. If Company_11 has a missing field, that field is filled with the median of the other companies (and we say so), rather than losing the row.


In [ ]:
TARGET = 'Company_11'
is_target = dataset['shortName'] == TARGET
assert is_target.sum() == 1, f'{TARGET} not found exactly once in the file'

num_cols = dataset.select_dtypes(exclude='object').columns
missing_target = dataset.loc[is_target, num_cols].isna().sum()
if missing_target.sum() > 0:
    print(f'{TARGET} has missing values in:', list(missing_target[missing_target > 0].index), '-> filled with column medians')
    dataset.loc[is_target, num_cols] = dataset.loc[is_target, num_cols].fillna(dataset.loc[~is_target, num_cols].median())

n_before = len(dataset)
dataset = dataset[is_target | dataset.notna().all(axis=1)].reset_index(drop=True)
print('rows before:', n_before, ' after dropping rows with missing values:', len(dataset))
is_target = dataset['shortName'] == TARGET        # recomputed: the index changed
assert is_target.sum() == 1


### Step 3.3: Which columns enter the clustering?

This is the decision that determines what "comparable" means. The cell below prints the columns we cluster on. Look at the list before you read on:

- **Level variables** (total revenue, total assets, EBITDA in currency units) measure size. If they dominate the feature set, k-means produces size buckets, and Company_11's "peers" are simply the firms with similar revenue, whatever their business.
- **Ratios** (margins, growth rates, returns, leverage, liquidity) measure the economics of a business. Comparable-company analysis wants firms with similar margins, growth and risk, because those are what a multiple prices.

We keep every numeric column here, including any levels, for two reasons: the notebook must run on the file as delivered, and it lets you see the problem in your own output. Exercise 3.1 asks you to redo the peer selection on ratios only and to compare the peer groups. If the printed list contains levels, treat the result below as a first pass, not a valuation.


In [ ]:
dataset_clustering = dataset.select_dtypes(exclude='object')
print('columns used for clustering (', dataset_clustering.shape[1], '):')
print(list(dataset_clustering.columns))

scaler_val = preprocessing.StandardScaler().fit(dataset_clustering)
X = pd.DataFrame(scaler_val.transform(dataset_clustering), columns=dataset_clustering.columns, index=dataset.index)


In [ ]:
# PCA for plotting only
pca = PCA(random_state=0).fit(X)
scores = pca.transform(X)
exp_var = pca.explained_variance_ratio_
print('variance explained by the first two components:', round(exp_var[:2].sum(), 3))

plt.figure(figsize=(10, 4))
plt.bar(range(1, len(exp_var) + 1), exp_var, alpha=0.5, label='individual')
plt.step(range(1, len(exp_var) + 1), np.cumsum(exp_var), where='mid', label='cumulative')
plt.xlabel('Principal component'); plt.ylabel('Explained variance ratio'); plt.legend()
plt.show()


### Step 3.4: Model selection

With a few dozen companies the number of clusters matters a lot: at K = 20 the average cluster has three firms, which is not a peer group. We sweep K from 2 to 12, keep every partition (`label_list`), and choose K from the table. The final clustering reuses the stored partition, so the K we inspected is the K we use.


In [ ]:
max_n_clusters = 12
rows = []
label_list = {}

fig, ax = plt.subplots(math.ceil((max_n_clusters - 1) / 3), 3, figsize=(16, 4 * math.ceil((max_n_clusters - 1) / 3)),
                       constrained_layout=True)
ax = ax.flatten()
for i in range(max_n_clusters - 1):
    nclust = i + 2
    sil, wcss, label, _ = cluster_kmeans(X, nclust)
    label_list[nclust] = label
    sizes = np.bincount(label)
    rows.append({'Clusters': nclust, 'Silhouette': sil, 'WCSS': wcss,
                 'smallest_cluster': sizes.min(), 'Company_11_peers': sizes[label[is_target.values][0]] - 1})
    cent = pd.DataFrame(scores).groupby(label).mean().values

    ax[i].scatter(scores[:, 0], scores[:, 1], c=label, cmap='Accent', s=40)
    ax[i].scatter(cent[:, 0], cent[:, 1], c='black', s=150, marker='P')
    ax[i].scatter(scores[is_target.values, 0], scores[is_target.values, 1], facecolors='none', edgecolors='red', s=250,
                  linewidths=2, label=TARGET)
    ax[i].set_title(f'K = {nclust}   sil = {sil:.3f}   WCSS = {wcss:.0f}', fontsize=13)
    ax[i].set_xlabel('PC1'); ax[i].set_ylabel('PC2')
ax[0].legend(loc='upper left', fontsize=9)
for j in range(max_n_clusters - 1, len(ax)):
    ax[j].axis('off')
plt.show()

tab = pd.DataFrame(rows)
display(tab.round(3))


In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(tab.Clusters, tab.Silhouette, 'x-', color='blue')
ax1.set_xlabel('Number of clusters', fontsize=14)
ax1.set_ylabel('Silhouette', fontsize=14, color='blue')
ax2 = ax1.twinx()
ax2.plot(tab.Clusters, tab.WCSS, 'x-', color='red')
ax2.set_ylabel('WCSS', fontsize=14, color='red')
plt.show()


**Choosing K.** Two constraints pull in opposite directions. The silhouette usually prefers a small K on data like this (companies do not form tight islands), but a small K makes the "peer group" most of the sample, which is no more comparable than the whole market. A large K gives small groups, and a `smallest_cluster` of 1 means one firm is so far from the rest (usually on a level variable) that it sits alone, another sign that size dominates the features.

The rule applied in the next cell: consider only the K for which `Company_11_peers` (the other firms in Company_11's cluster, not counting Company_11 itself) is at least `MIN_PEERS` (5) and at most a third of the sample, and among those take the K with the highest silhouette. The same count, with the same threshold, is checked again when the peer group is built, so a K that passes here cannot fail there. If no K qualifies, fall back to the best silhouette overall. The cell prints the candidates and the K it picked. Override `K` by hand if the table suggests a better trade-off for your data, and say why in one sentence.


In [ ]:
MIN_PEERS = 5
candidates = tab[(tab['Company_11_peers'] >= MIN_PEERS) & (tab['Company_11_peers'] <= len(dataset) / 3)]
print('candidate K (peer group between', MIN_PEERS, 'and', len(dataset) // 3, 'firms):')
display(candidates.round(3))
if len(candidates):
    K = int(candidates.loc[candidates['Silhouette'].idxmax(), 'Clusters'])
else:
    K = int(tab.loc[tab['Silhouette'].idxmax(), 'Clusters'])
    print('no K gives a peer group of that size; falling back to the best silhouette')
print('K chosen from the table:', K)

# Reuse the partition from the sweep (same seed, same n_init): what we inspected is what we use
dataset['Cluster'] = label_list[K]
print('cluster sizes:', np.bincount(dataset['Cluster']))


### Step 3.5: Company_11's peer group


In [ ]:
Company11_cluster = int(dataset.loc[dataset['shortName'] == TARGET, 'Cluster'].iloc[0])
similar_companies = dataset[(dataset['Cluster'] == Company11_cluster) & (dataset['shortName'] != TARGET)]

n_peers = similar_companies.shape[0]
print(f'{TARGET} is in cluster {Company11_cluster}, which holds {n_peers} other companies')
if n_peers < MIN_PEERS:
    raise ValueError(f'Only {n_peers} peers: a multiple from fewer than {MIN_PEERS} firms is not a valuation. '
                     'Lower K in the cell above or change the clustering features.')
similar_companies


In [ ]:
# Market data for the public companies, merged onto the peer group
data_extra = pd.read_csv(RAW + 'data/financialdata_extra.csv')
print('market data columns:', list(data_extra.columns))

merged_data = pd.merge(similar_companies, data_extra, on='shortName', how='left')
print('peers without market data (dropped):', int(merged_data['enterpriseValue'].isna().sum()))
merged_data = merged_data.dropna(subset=['enterpriseValue'])
merged_data[['shortName', 'ebitda', 'enterpriseValue', 'marketCap']]


### Step 3.6: Multiples and valuation

Three rules before computing anything:

1. **Positive EBITDA only.** A negative or near-zero EBITDA gives a negative or exploding EV/EBITDA that says nothing about value. Such peers are excluded from the multiple.
2. **Median, not mean**, with the interquartile range as the valuation range.
3. **Peer count is part of the answer.** Report how many multiples the median is based on.


In [ ]:
peers = merged_data[merged_data['ebitda'] > 0].copy()
print('peers with market data:', len(merged_data), '  of which EBITDA > 0:', len(peers))
if len(peers) < MIN_PEERS:
    raise ValueError(f'Only {len(peers)} peers with positive EBITDA; not enough for a multiple.')

peers['EV_to_ebitda'] = peers['enterpriseValue'] / peers['ebitda']
print('\nEV/EBITDA across the peers:')
print(peers['EV_to_ebitda'].describe().round(2))

median_multiple = peers['EV_to_ebitda'].median()
q25, q75 = peers['EV_to_ebitda'].quantile([0.25, 0.75])
print(f'\nmedian EV/EBITDA: {median_multiple:.2f}x   interquartile range: {q25:.2f}x to {q75:.2f}x   (n = {len(peers)})')
print(f'for comparison, the mean: {peers["EV_to_ebitda"].mean():.2f}x')


In [ ]:
ebitda_value_company11 = dataset.loc[dataset['shortName'] == TARGET, 'ebitda'].values[0]
if ebitda_value_company11 <= 0:
    raise ValueError(f'{TARGET} has EBITDA {ebitda_value_company11:,.0f}; an EV/EBITDA multiple cannot be applied.')

Company11_EV = median_multiple * ebitda_value_company11
Company11_EV_low = q25 * ebitda_value_company11
Company11_EV_high = q75 * ebitda_value_company11

print(f'{TARGET} EBITDA:            {ebitda_value_company11:,.0f}')
print(f'estimated EV (median multiple): {Company11_EV:,.0f}')
print(f'range (25th to 75th pct multiple): {Company11_EV_low:,.0f} to {Company11_EV_high:,.0f}')
print(f'based on {len(peers)} peers')

# Cross-check with EV/Sales if the file has a revenue column
revenue_col = next((c for c in ['totalRevenue', 'revenue', 'Revenue', 'sales', 'Sales'] if c in dataset.columns), None)
if revenue_col is not None:
    peers['EV_to_sales'] = peers['enterpriseValue'] / peers[revenue_col]
    sales_11 = dataset.loc[dataset['shortName'] == TARGET, revenue_col].values[0]
    ev_sales = peers['EV_to_sales'].median() * sales_11
    print(f'\ncross-check, median EV/{revenue_col} = {peers["EV_to_sales"].median():.2f}x  ->  EV = {ev_sales:,.0f}')
    print(f'ratio of the two estimates (EV/Sales over EV/EBITDA): {ev_sales / Company11_EV:.2f}')
else:
    print('\nno revenue column found for an EV/Sales cross-check; add one if the file has it')


**Reading the result.** The valuation is a median multiple, a range, and a peer count. If the range is wide (the 75th percentile multiple is more than twice the 25th), the peers disagree about what a unit of EBITDA is worth, which usually means the peer group mixes businesses that are not comparable; go back to Step 3.3. If the EV/Sales cross-check is far from the EV/EBITDA estimate, Company_11's margin is unlike its peers', and the peer group again deserves a second look.

### Part 3 summary

**What we did:** clustered companies on their fundamentals, took Company_11's cluster as its peer group, checked the group was large enough, and valued Company_11 at the median peer EV/EBITDA with an interquartile range.

**Advantages of clustering-based peer selection:** repeatable and auditable; uses many financial dimensions at once; not limited to industry codes.

**Practical considerations:**

- **Data quality.** Garbage in, garbage out. Fundamentals must be on a comparable basis (IFRS versus GAAP, fiscal year ends, one-off items in EBITDA).
- **Which features.** Levels create size buckets; ratios create economic peers. Print the list and decide deliberately.
- **Market conditions.** Multiples reflect current sentiment; consider the point in the cycle, and remember that private companies usually trade at a discount for illiquidity.
- **Validate the peers.** Always look at the names in the peer group and ask whether an analyst would accept them. Clustering proposes; judgement disposes.

### Exercises, Part 3

1. **Ratios only.** Rebuild `dataset_clustering` from ratio columns only (margins, growth, returns, leverage, liquidity; drop revenue, EBITDA and other currency amounts), rerun Steps 3.3 to 3.6. Deliverable: the two peer lists (levels versus ratios), how many names they share, and the two EV estimates with one sentence on which peer group you trust more and why.
2. **Sensitivity to K.** Set `K` by hand to 3, 5 and 8 and rerun Steps 3.5 and 3.6. Deliverable: a table with K, peer count, median multiple and EV estimate, and one sentence on how the valuation moves with K.
3. **Mean versus median.** Add one artificial peer with an EV/EBITDA of 60 to `peers` and recompute both the mean and the median multiple. Deliverable: the two multiples before and after, and one sentence on which you would put in a valuation report.


---

# Overall Summary

### What we learned

**Part 1.** K-means partitions data into K compact groups and always returns K groups; the silhouette and WCSS help choose K but reward compactness, not truth. DBSCAN follows density and needs no K, at the price of two parameters that must fit the data.

**Part 2.** Clustering as a pre-modelling step is a hypothesis: that default drivers differ across segments by more than the cost of fitting each model on fewer rows. Testing it requires one split, one scaler, clusters fitted on training rows, and every model evaluated on the same test rows against the global model. The result of that test is the lesson, whichever way it comes out.

**Part 3.** Clustering can propose comparable companies, but the valuation is only as good as the feature list that defines "comparable", the size of the peer group, and the choice of a robust multiple (median with a range).

### Choosing a clustering method

| Situation | Method | Why |
|---|---|---|
| Compact, roughly spherical groups, K known or chosen by a sweep | k-means | fast, easy to read, returns centroids |
| Arbitrary shapes, outliers to be flagged, K unknown | DBSCAN | follows density, labels noise |
| Small data, want the full merge hierarchy | hierarchical | dendrogram shows structure at every K |
| Very large data | k-means or mini-batch k-means | scales linearly |

### Before you deploy a clustering

- Standardise features, and decide deliberately which features define similarity.
- Fit the scaler and the clustering on training data; assign new observations with `predict`.
- Check that clusters make business sense and are stable over time.
- If clusters feed a downstream model, compare against the model without them, on common test rows.

### Further reading

- Hastie, Tibshirani and Friedman, *The Elements of Statistical Learning*, chapter 14 (unsupervised learning)
- scikit-learn user guide: [Clustering](https://scikit-learn.org/stable/modules/clustering.html)
- Damodaran, *Investment Valuation*, the chapters on relative valuation (choice of comparables and of the multiple)
